# Train Spectron MoE + HRM on a Colab T4 (Vulkan, Tensor Cores)

Choose **Runtime → Change runtime type → T4 GPU**, then run the cells from top to bottom. The notebook installs Nim 2.2.10 and Vulkan, clones the repository with its submodule, downloads the 5-billion-token corpus with `hf download` (`make download-datasets`, about 6.25 GB), builds the EOS data (`make prepare-spectron-eos-data`), builds the `test_nimformer` trainer with `-d:backend=vulkan`, and keeps training the MoE model `finetune.nimq` with its HRM core and EOS token.

- **Tensor Cores:** the trainer runs with `--tensor-cores=require`. At startup it runs a 64×64 GEMM through `KHR_cooperative_matrix` (FP16 inputs, FP32 accumulation) and checks the result. If the driver lacks support, the trainer stops instead of silently running FP32.
- **HRM:** the 2 Spectron blocks run as a Hierarchical Reasoning Model ([arXiv:2506.21734](https://arxiv.org/abs/2506.21734)): block 0 is the low-level module L and block 1 the high-level module H. Each block has 2 experts and its own top-1 router.
- **Batch:** `BATCH = 256` by default (cell 5).
- **Checkpoint:** every 20 steps the trainer saves `test/spectron/models/finetune.nimq`, and the training cell uploads the newest save to [bobbydeveloper2014/bybyai-spectron](https://huggingface.co/bobbydeveloper2014/bybyai-spectron) as `finetune.nimq` at the repo root with `hf upload` while training continues. After a disconnect, rerun the cells from the top: the download cell fetches `finetune.nimq` with `hf download`, and the trainer resumes with its Adam state.

**Hugging Face token:** cell 5 mounts Google Drive and reads a write token from `MyDrive/.env_spectron_token_huggingface`. If the file is missing, the cell asks for the token (hidden input) and saves it there with mode 600. The token is never printed and never stored in the notebook. To change it, delete that file and rerun cell 5.

In [ ]:
# 1) Check the GPU runtime and install system packages, including the Vulkan runtime and the shader compiler (glslc).
import os, subprocess, sys
from pathlib import Path

assert Path('/content').exists(), 'This notebook must run on Google Colab.'
try:
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'], text=True))
except Exception as e:
    raise RuntimeError('No NVIDIA GPU. Choose Runtime → Change runtime type → T4 GPU and run again.') from e

subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq','build-essential','git','curl','ca-certificates','xz-utils','pkg-config','time','libvulkan1','libvulkan-dev','vulkan-tools','glslc','libegl1','libglx0','libx11-6','libxext6','libx11-xcb1','libxcb1','libxcb-dri3-0','libxcb-present0','libxfixes3','libxrandr2','libdrm2','libexpat1','libwayland-client0'], check=True)

In [ ]:
# 2) Install the official Nim 2.2.10 release and verify its checksum.
import hashlib, urllib.request, tarfile, os, subprocess
from pathlib import Path

NIM_VERSION = '2.2.10'
TOOLS = Path('/content/tools'); TOOLS.mkdir(exist_ok=True)
archive = TOOLS / f'nim-{NIM_VERSION}-linux_x64.tar.xz'
url = f'https://nim-lang.org/download/nim-{NIM_VERSION}-linux_x64.tar.xz'

# A browser User-Agent avoids a 403 from the download server.
if not archive.exists():
    req = urllib.request.Request(
        url,
        headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
    )
    with urllib.request.urlopen(req) as response, open(archive, 'wb') as out_file:
        out_file.write(response.read())

# Verify the checksum.
expected = '0a3a38752e97e9d44aa479b3a7b37336dfe0176daf22ee5b5218ad0991ecd211'
actual = hashlib.sha256(archive.read_bytes()).hexdigest()
assert actual == expected, f'Nim checksum mismatch: {actual}'

# Extract.
with tarfile.open(archive, 'r:xz') as tf:
    tf.extractall(TOOLS, filter='data')

# Add Nim to PATH.
NIM_HOME = TOOLS / f'nim-{NIM_VERSION}'
os.environ['PATH'] = f"{NIM_HOME}/bin:" + os.environ['PATH']

# Check the version.
subprocess.run(['nim', '--version'], check=True)

In [ ]:
# 3) Clone the latest main branch and fetch the submodule pinned by the repository.
import os, subprocess
from pathlib import Path

URL = 'https://github.com/bobbyshop-vui/nimformers-framework.git'
BRANCH = 'main'
REPO = Path('/content/nimformers-framework')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
subprocess.run(['git', '-C', str(REPO), 'submodule', 'sync', '--recursive'], check=True)
subprocess.run(['git', '-C', str(REPO), 'submodule', 'update', '--init', '--recursive'], check=True)
os.chdir(REPO)
print('Repo:', REPO, '| branch:', BRANCH)
print(subprocess.check_output(['git', 'log', '--oneline', '-1'], text=True))

In [ ]:
# 4) Install Vulkan, check the NVIDIA ICD and confirm a discrete GPU (lavapipe/CPU is rejected).
import subprocess, os, re
from pathlib import Path

# Step 1: install vulkan-tools.
print("[1/4] Installing vulkan-tools...")
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'vulkan-tools'], check=True)

# Step 2: detect the NVIDIA driver version and install the matching libnvidia-gl.
print("[2/4] Detecting the NVIDIA driver version...")
smi = subprocess.check_output(['nvidia-smi'], text=True)
m = re.search(r'Driver Version:\s*(\d+)\.', smi)
assert m, "Could not read the driver version from nvidia-smi"
driver_major = m.group(1)
print(f"    -> Driver major: {driver_major}")

pkg = f'libnvidia-gl-{driver_major}'
print(f"[2/4] Installing {pkg}...")
subprocess.run(['apt-get', 'install', '-y', '-qq', pkg], check=True)

# Step 3: make sure the Vulkan loader can find the NVIDIA ICD file.
print("[3/4] Configuring the Vulkan ICD...")
Path('/usr/share/vulkan/icd.d').mkdir(parents=True, exist_ok=True)
candidates = [
    '/etc/vulkan/icd.d/nvidia_icd.json',
    '/usr/share/vulkan/icd.d/nvidia_icd.json',
    f'/usr/lib/x86_64-linux-gnu/libnvidia-gl-{driver_major}/nvidia_icd.json',
]
src = next((c for c in candidates if Path(c).exists()), None)
if src and src != '/usr/share/vulkan/icd.d/nvidia_icd.json':
    subprocess.run(['ln', '-sf', src, '/usr/share/vulkan/icd.d/nvidia_icd.json'], check=True)
    print(f"    -> symlink: {src} -> /usr/share/vulkan/icd.d/nvidia_icd.json")
else:
    print(f"    -> ICD already present: {src}")

# Step 4: run the check.
print("[4/4] Checking Vulkan...")
subprocess.run(['vulkaninfo', '--summary'], check=True,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
summary = subprocess.check_output(['vulkaninfo', '--summary'],
                                   text=True, stderr=subprocess.STDOUT)
print(''.join(line for line in summary.splitlines()
              if any(k in line.lower() for k in ('device name', 'device type', 'driver name'))))

assert 'NVIDIA' in summary.upper(), \
    'No NVIDIA Vulkan ICD. This GPU runtime does not expose Vulkan graphics.'
assert 'DISCRETE_GPU' in summary.upper(), \
    'No NVIDIA discrete GPU through Vulkan; stopping to avoid a CPU/iGPU fallback.'

print("Vulkan OK: the NVIDIA discrete GPU is exposed.")

In [ ]:
# 5) Run settings and where the Hugging Face token is kept.
import os, getpass
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
# The trainer's Hugging Face mode reads the write token from this file (--hf-token-file).
TOKEN_FILE = Path('/content/drive/MyDrive/.env_spectron_token_huggingface')
if TOKEN_FILE.is_file() and TOKEN_FILE.read_text().strip():
    print('Token: found in', TOKEN_FILE)
else:
    token = getpass.getpass('Hugging Face write token (input is hidden): ').strip()
    assert token, 'No token entered.'
    TOKEN_FILE.write_text(token + '\n')
    TOKEN_FILE.chmod(0o600)
    del token
    print('Token: saved to', TOKEN_FILE)

HF_REPO = 'bobbydeveloper2014/bybyai-spectron'
HF_CHECKPOINT = 'finetune.nimq'
DATASET = REPO / 'test/spectron/data/scale5b'
CHECKPOINT = REPO / 'test/spectron/models/finetune.nimq'
TRAINER = REPO / 'test/spectron/test_nimformer'
EXPERTS = 2         # 2 experts in each of the 2 blocks, 33,088 parameters each.
SEQ = 128
SAVE_EVERY = 20     # Save every 20 optimizer steps; each save is uploaded.
STEPS = 100_000     # Additional steps for this session; rerun to continue.
BATCH = 256

# HRM core: 1 low-level (L) and 1 high-level (H) block; the checkpoint keeps its layout.
HRM_CYCLES = 2      # High-level cycles per segment.
HRM_L_STEPS = 2     # Low-level updates per cycle.
HRM_SEGMENTS = 2    # Deep-supervision segments per optimizer step.
HRM = [f'--hrm-cycles={HRM_CYCLES}', f'--hrm-l-steps={HRM_L_STEPS}',
       f'--hrm-segments={HRM_SEGMENTS}']
print('HRM:', ' '.join(HRM))
print('Batch:', BATCH)
print('Checkpoint:', CHECKPOINT, '->', f'{HF_REPO}/{HF_CHECKPOINT}')

In [ ]:
# 6) Download the packed corpus (about 6.25 GB), build the EOS data, and fetch finetune.nimq from the Hub root.
import os, shutil, subprocess
os.chdir(str(REPO))
subprocess.run(['pip', 'install', '-q', '-U', 'huggingface_hub', 'pyarrow'], check=True)
subprocess.run(['make', 'download-datasets'], check=True)
if not (DATASET / 'eos/manifest.json').exists():
    subprocess.run(['hf', 'download', 'databricks/databricks-dolly-15k', 'databricks-dolly-15k.jsonl',
                    '--repo-type', 'dataset', '--local-dir', str(REPO / 'test/spectron')], check=True)
    subprocess.run(['make', 'prepare-spectron-eos-data'], check=True)
if CHECKPOINT.exists():
    print('Using local', CHECKPOINT)
else:
    tmp = REPO / 'test/spectron/models/.hf-restore'
    shutil.rmtree(tmp, ignore_errors=True)
    subprocess.run(['hf', 'download', HF_REPO, HF_CHECKPOINT, '--local-dir', str(tmp)], check=True)
    CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    (tmp / HF_CHECKPOINT).replace(CHECKPOINT)
    shutil.rmtree(tmp, ignore_errors=True)
    print('Restored', CHECKPOINT)

In [ ]:
# 7) Compile the Vulkan shaders and the Spectron trainer for the Vulkan backend.
import os
os.chdir(str(REPO))
!make shaders
!make build-spectron-trainer SPECTRON_TRAIN_BACKEND=vulkan
assert TRAINER.is_file(), f'Trainer not found: {TRAINER}'

In [ ]:
# 8) Train in Hugging Face mode: the trainer uploads every save with `hf upload` while it trains.
import os, signal, subprocess

os.chdir(str(REPO))
command = [
    TRAINER, f'--dataset={DATASET}', f'--checkpoint={CHECKPOINT}',
    f'--moe-experts={EXPERTS}', f'--batch={BATCH}', f'--seq={SEQ}', f'--steps={STEPS}',
    f'--checkpoint-every={SAVE_EVERY}', f'--log-every={SAVE_EVERY}',
    f'--hf-repo={HF_REPO}', f'--hf-path={HF_CHECKPOINT}', f'--hf-token-file={TOKEN_FILE}',
    '--tensor-cores=require', *HRM,
]
print(' '.join(str(part) for part in command), flush=True)
process = subprocess.Popen([str(part) for part in command], cwd=str(REPO),
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, errors='replace')
try:
    for line in process.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    print('Stopping the trainer; the last saved checkpoint is kept.', flush=True)
    process.send_signal(signal.SIGTERM)
    for line in process.stdout:
        print(line, end='', flush=True)
finally:
    print('Trainer exit code:', process.wait())

### Notes

- `[train]` lines show `step`, `epoch`, `loss`, `tok/s` and the number of trained tokens. `[val]` lines show the code and English validation losses every 500 steps. `[hf-upload]` lines show when an upload starts and finishes.
- The `[hrm]` line at startup prints the L and H block counts, cycles, L steps, segments and block passes per segment.
- Each upload is one commit on Hugging Face that replaces `finetune.nimq` at the repo root with the whole checkpoint: weights, block routers and experts, the HRM LayerNorms and the Adam state of every parameter.
- `EXPERTS` must match the checkpoint (2 experts per block); the trainer stops on a mismatch.

### Documentation

- [Spectron MoE training guide](https://github.com/bobbyshop-vui/nimformers-framework/blob/main/docs/spectron-guide.md)
- [Google Colab FAQ: GPU runtime](https://research.google.com/colaboratory/faq.html)
- [VK_KHR_cooperative_matrix](https://registry.khronos.org/vulkan/specs/latest/man/html/VK_KHR_cooperative_matrix.html)